# Initializations

In [ ]:
!apt-get update && apt-get install -y build-essential 1>/dev/null

In [ ]:
!apt-get update && apt-get install -y jq 1>/dev/null

In [ ]:
!pip install --upgrade pip  1>/dev/null

## Requirements

In [ ]:
!pip install langchain==0.0.230 1>/dev/null

In [ ]:
!pip install openai==0.27.8 1>/dev/null

In [ ]:
!pip install faiss-cpu==1.7.4  1>/dev/null

In [ ]:
!pip install tiktoken==0.4.0 1>/dev/null

In [ ]:
!pip install unstructured==0.8.8 1>/dev/null

## Spacy (NLP)

In [ ]:
!pip install spacy 1>/dev/null
# https://spacy.io/usage/spacy-101
# https://realpython.com/natural-language-processing-spacy-python/
# https://spacy.io/usage/spacy-101

In [ ]:
!python -m spacy download en_core_web_sm 1>/dev/null

## Secrets and credentials

In [ ]:
%%bash --out secrets 
# using AWS's Secret Manager to store keys
# garb the keys and store it into a Pytthon variable
export RESPONSE=$(aws secretsmanager get-secret-value --secret-id 'salvia/labbench/tests' )
export SECRETS=$( echo $RESPONSE | jq '.SecretString | fromjson')

echo $SECRETS

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = eval(secrets)["OPENAI_API_KEY"]


# Download document

In [ ]:
!mkdir -p work/llmops/data

In [ ]:
# https://archive.org/download/in.ernet.dli.2015.136553

In [ ]:
!curl https://ia902905.us.archive.org/33/items/in.ernet.dli.2015.136553/2015.136553.Alice-In-Wonderland_text.pdf \
 -o work/llmops/data/alice_in_wonderland.pdf

In [ ]:
!ls -l work/llmops/data

In [ ]:
notebook_folder = "work/llmops"
documents_folder = f"{notebook_folder}/data"

print(documents_folder)

# Load document

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.document_loaders import (
    TextLoader,
    UnstructuredPDFLoader,
    UnstructuredPowerPointLoader,
    UnstructuredWordDocumentLoader,
)

# This is the source document.    
document_path = f"{documents_folder}/alice_in_wonderland.pdf"
 
# Setup a text loader
#loader = TextLoader(document_path)
loader = UnstructuredPDFLoader(document_path)

alice_documents = loader.load()
print(f"Loaded {len(alice_documents)} documents")
print(f"Document size is {len(alice_documents[0].page_content)} characters")

# Split Document

In [ ]:
class SplitterStrategy:
    def __init__(self):
        self.splitter = None
        
    def get_splitter(self):
        return self.splitter

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

class CharacterTextSplitterStrategy(SplitterStrategy):
    def __init__(self, splitter_parameters):
        # Get your splitter ready
        self.splitter = CharacterTextSplitter(
                            #separator = "\n\n",
                            chunk_size=splitter_parameters['chunk_size'], 
                            chunk_overlap=splitter_parameters['chunk_overlap'])

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

class RecursiveCharacterTextSplitterStrategy(SplitterStrategy):
    def __init__(self, splitter_parameters):
        # Get your splitter ready
        self.splitter = RecursiveCharacterTextSplitter(
                            chunk_size=splitter_parameters['chunk_size'], 
                            chunk_overlap=splitter_parameters['chunk_overlap'])

In [ ]:
from langchain.text_splitter import SpacyTextSplitter

class SpacyTextSplitterStrategy(SplitterStrategy):
    def __init__(self, splitter_parameters):
        # Get your splitter ready
        # split by tokens
        self.splitter = SpacyTextSplitter(chunk_size=splitter_parameters['chunk_size'])

In [ ]:
splitter_parameters = {
    'chunk_size': 2000,
    'chunk_overlap': 100,
}

In [ ]:
#text_splitter_service = CharacterTextSplitterStrategy(splitter_parameters)
#text_splitter_service= RecursiveCharacterTextSplitterStrategy(splitter_parameters)
text_splitter_service = SpacyTextSplitterStrategy(splitter_parameters)

# Split your docs into texts
texts = text_splitter_service.get_splitter().split_documents(alice_documents)
print(f"Splitted into {len(texts)} parts")

In [ ]:
# add ids in metadata
i = 0
for text in texts:
    i += 1
    text.metadata['id'] = i
    
print(f"Metadata of first part {texts[0].metadata}")

# Setup the embedding model

In [ ]:
class EmbeddingsStrategy:
    def __init__(self):
        self.model = None
        
    def get_model(self):
        return self.model

In [ ]:
from langchain.embeddings import FakeEmbeddings

class FakeEmbeddingsStrategy(EmbeddingsStrategy):
    def __init__(self):
        self.model = FakeEmbeddings(size=1352)

In [ ]:
from langchain.embeddings.spacy_embeddings import SpacyEmbeddings

class SpacyEmbeddingsStrategy(EmbeddingsStrategy):
    def __init__(self):
         self.model = SpacyEmbeddings()

In [ ]:
from langchain.embeddings import OpenAIEmbeddings

class OpenAIEmbeddingsStrategy(EmbeddingsStrategy):
    def __init__(self, embeddings_parameters):
        self.model = OpenAIEmbeddings(
            model=embeddings_parameters["model_name"],
            chunk_size=1,
        )

In [ ]:
embeddings_parameters = {
    'model_name': "text-embedding-ada-002",
}

#embeddings_service = FakeEmbeddingsStrategy()
embeddings_service = SpacyEmbeddingsStrategy()
#embeddings_service = OpenAIEmbeddingsStrategy(embeddings_parameters)

embeddings = embeddings_service.get_model()

In [ ]:

text = alice_documents[0].page_content

# check the length
length = len(text)
print(f"{length=}")

try:
    # check the number of tokens
    num_tokens = embeddings.get_num_tokens(text)
    print(f"{num_tokens=}")
except:
    pass


# Index documents

In [ ]:
#from langchain.vectorstores import Annoy
from langchain.vectorstores import FAISS

# Embedd your texts andd store them in the vector database
# dtabase is in memory. it might be savecd to a file and loader later on.
#db = Annoy.from_documents(texts, embeddings)
vectorstore = FAISS.from_documents(texts, embeddings)

In [ ]:
print(f"index size {len(vectorstore.docstore._dict.keys())}")

# Retrieve documents 

In [ ]:
class SearchStrategy:
    def __init__(self):
        self.retriever = None
        
    def get_retriever(self):
        return self.retriever

In [ ]:
class RetrieverSearchStrategy(SearchStrategy):
    def __init__(self, vectorstore, search_type, search_parameters):
        self.retriever = vectorstore.as_retriever(
                            search_type=search_type, 
                            search_kwargs=search_parameters)

In [ ]:
retriever_parameters = {
    'k': 5,
    # close to 0 means very similar
    'score_threshold': 0.9
}

In [ ]:
# Init a retriever for this db
# lookup for some relevqnt parts

#search_service = RetrieverSearchStrategy(vectorstore, "similarity", retriever_parameters)
search_service = RetrieverSearchStrategy(vectorstore, "mmr", retriever_parameters)
#search_service = RetrieverSearchStrategy(vectorstore, "similarity_score_threshold", retriever_parameters)

retriever = search_service.get_retriever()

In [ ]:
# retrieve some indexed documents relevant for this query
query = "White Rabbit"
docs = retriever.get_relevant_documents(query)
print(f"Found {len(docs)}")

docs_ids = [doc.metadata['id'] for doc in docs]
print(f"Ids {docs_ids}")

assessment = [ 1 if query in doc.page_content else 0
              for doc in docs]
print(f"assessment {sum(assessment)}")

#samples = "\n\n".join([x.page_content[:200] for x in docs[:5]])
#print(samples)

# Search documents with FAISS API

In [ ]:
k = retriever_parameters['k']
score_threshold = retriever_parameters['score_threshold']
query = "White Rabbit"
results_with_scores = vectorstore.similarity_search_with_score(
    query,
    k=k,
    score_threshold=score_threshold
)
print(f"Found {len(results_with_scores)}")

results_ids = [(result[0].metadata['id'], result[1]) for result in results_with_scores]
print(f"Ids {results_ids}")


assessment = [ 1 if query in result[0].page_content else 0
              for result in results_with_scores]
print(f"assessment {sum(assessment)}")


# Evaluation of a query

In [ ]:
# TODO 2 QAstrategies + 2 lists as fakes

In [ ]:
class LLMStrategy:
    def __init__(self):
        self.model = None
        
    def get_model(self):
        return self.model

In [ ]:
from langchain.llms.fake import FakeListLLM

class FakeLLMStrategy(LLMStrategy):
    def __init__(self, responses):
        self.responses = responses
        self.model = None
        
    def get_model(self):
        self.model = FakeListLLM(responses=self.responses)
        return self.model


In [ ]:
from langchain.llms import OpenAI

class OpenAILLMStrategy(LLMStrategy):
    def __init__(self, llm_parameters):
        self.model = OpenAI(temperature=llm_parameters['temperature'], 
                             model_name=llm_parameters['model_name'])
 

In [ ]:
llm_parameters = {
    # Note, the default model is already 'text-davinci-003' 
    'model_name': "text-davinci-003",
    # temperature 0 means no randomness
    'temperature': 0
}

In [ ]:
fake_responses = ["White Rabbit is always late"]

In [ ]:
from langchain.chains import RetrievalQA

llm_service = FakeLLMStrategy(fake_responses)
#llm_sergice = OpenAILLMStrategy(llm_parameters)

llm_model = llm_service.get_model()

# Asking theLLM
# the response will be based on the retrieved documents 
qa_chain = RetrievalQA.from_chain_type(llm_model, 
                                 chain_type="stuff", 
                                 retriever=retriever)

query = "White Rabbit"
response = qa_chain.run(query)
print(f"{response=}")

# Evaluation of a bunch of queries

In [ ]:
question_answers = [
    {'question' : "Who is the protagonist?", 
     'answer' : "Alice"},
    {'question' : "Who is Alice?", 
     'answer' : 
        """Alice is the curious young girl who goes on a wild
        adventure full of strange and wonderful creatures."""},
    {'question' : "Who is the Caterpilar?", 
     'answer' : 
        """Caterpillar is an old wise sage who has been
        around for hundreds of years. He is a mysterious
        creature with an ever-changing form."""},
    {'question' : "Who is the Chesshire Cat?", 
     'answer' : 
        """The Cheshire Cat is a mysterious and mischievous
        creature who loves to play pranks on Alice. He
        is fond of disappearing and reappearing,"""},
    {'question' : "Who is the Mad Hatter?", 
     'answer' : 
        """The Mad Hatter loves to throw mad tea-parties with
        his friends, the March Hare and the Dormouse. He is
        always dressed in the craziest outfits, and loves
        to talk in riddles and ask nonsensical questions."""},
    {'question' : "Who is the Queen of Hearts?", 
     'answer' : 
        """The Queen of Hearts rules the land with an iron fist. She is
        known for her grandiose parties, where she can often be
        found shouting orders and demanding absolute obedience.
        Her favorite game is croquet, a game she plays with a
        rather unorthodox set of rules that only she understands."""},
    {'question' : "Who is the White Rabbit?", 
     'answer' : 
        """White Rabbit is an anxious, time-obsessed
        character from Alice in Wonderland who is always
        running late and is constantly in a hurry."""},
]
print(f"There are {len(question_answers)} question answers")

In [ ]:
fake_responses = [item['answer'] for item in question_answers]
print(f"There are {len(fake_responses)} fake responses")

In [ ]:
from langchain.chains import RetrievalQA

llm_service = FakeLLMStrategy(fake_responses)
#llm_sergice = OpenAILLMStrategy(llm_parameters)

llm_model = llm_service.get_model()

# Asking theLLM
# the response will be based on the retrieved documents 
qa_chain = RetrievalQA.from_chain_type(llm_model, 
                                 chain_type="stuff", 
                                 retriever=retriever,  
                                 input_key="question")


In [ ]:
from pprint import pprint

predictions = qa_chain.apply(question_answers)
    
print(f"There are {len(predictions)} predictions")
pprint(predictions)

# Evaluate answers

In [ ]:
fake_responses = ['CORRECT' for item in question_answers]
fake_eval = ( retriever_parameters['k'] > 5 and 
        retriever_parameters['score_threshold'] > 0.9 and 
        splitter_parameters['chunk_size'] <= 2000 )           
fake_responses[0] = 'CORRECT' if fake_eval else 'INCORRECT' 
print(f"There are {len(fake_responses)} responses")

In [ ]:
from langchain.evaluation.qa import QAEvalChain


llm_service = FakeLLMStrategy(fake_responses)
#llm_sergice = OpenAILLMStrategy(llm_parameters)

llm_model = llm_service.get_model()

# Start your eval chain
eval_chain = QAEvalChain.from_llm(llm_model)

# Have it grade itself. The code below helps the eval_chain know where the different parts are
graded_outputs = eval_chain.evaluate(question_answers,
                                     predictions,
                                     question_key="question",
                                     prediction_key="result",
                                     answer_key='answer')
graded_outputs

# Retriever optimization

TODO optimize for a list of queries

In [ ]:
!pip install optuna 1>/dev/null

In [ ]:
# https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html

In [ ]:
search_types = ["similarity", "mmr",  "similarity_score_threshold"]

In [ ]:
from functools import partial

# function to optimize
def get_retriever(vectorstore, search_type, k, score_threshold):
    retriever_parameters = {
        'k': k,
        # close to 0 means very similar
        'score_threshold': score_threshold
    }
    search_service = RetrieverSearchStrategy(vectorstore, "mmr", retriever_parameters)
    
    return search_service.get_retriever()

# currying partial
get_retriever_ftom_vectorstore = partial(get_retriever, vectorstore)

In [ ]:
retriever = get_retriever_ftom_vectorstore(search_types[0], 5, 0.3)
# retrieve some indexed documents relevant for this query
query = "White Rabbit"
docs = retriever.get_relevant_documents(query)
print(f"Found {len(docs)}")

In [ ]:
# function scoring
def evaluate_retriever(retriever):
    query = "White Rabbit"
    search_results = retriever.get_relevant_documents(query)

    # how many documents actually contins the query
    assessment = [ 1 if query in result.page_content else 0
                  for result in search_results]
    return sum(assessment)

In [ ]:
score = evaluate_retriever(retriever)
print(f"{score=}")

In [ ]:
import optuna
from optuna.study import StudyDirection

# function to be minimized
def objective(trial):
    search_type = trial.suggest_categorical('search_type', search_types)
    k = trial.suggest_int('k', 1, 20)
    score_threshold = trial.suggest_float('score_threshold', 0.0, 1.0)
    return evaluate_retriever(get_retriever_ftom_vectorstore(search_type, k, score_threshold))

study = optuna.create_study(direction=StudyDirection.MAXIMIZE)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100)

study.best_params

# Document search optimization

idem plus splitter configuration

TODO study  name

In [ ]:
from functools import partial

# function to optimize

configuration = {
    'splitter': {
        'splitter_services': {
            'charecter': CharacterTextSplitterStrategy,
            'recursive': RecursiveCharacterTextSplitterStrategy,
            'token': SpacyTextSplitterStrategy
        }
    },
    'search': {
        'search_types': ["similarity", "mmr",  "similarity_score_threshold"]
    }
}

# function to optimize
def get_splitter(service_cls, chunk_size, chunk_overlap):
    splitter_parameters = {
        'chunk_size': chunk_size,
        'chunk_overlap': chunk_overlap,
    }
    search_service = service_cls(splitter_parameters)
    return search_service.get_retriever()

def get_retriever(vectorstore, search_type, k, score_threshold):
    retriever_parameters = {
        'k': k,
        # close to 0 means very similar
        'score_threshold': score_threshold
    }
    search_service = RetrieverSearchStrategy(vectorstore, "mmr", retriever_parameters)
    
    return search_service.get_retriever()

def get_processor(documents, service_cls, chunk_size, chunk_overlap, search_type, k, score_threshold):
    texts = text_splitter_service.get_splitter().split_documents(documents)

    embeddings_service = SpacyEmbeddingsStrategy()
    embeddings = embeddings_service.get_model()
    vectorstore = FAISS.from_documents(texts, embeddings)

    retriever = get_retriever(vectorstore,  search_type, k, score_threshold)
    return retriever

# currying partial
get_processor_from_documennts = partial(get_processor, alice_documents)


In [ ]:
# function scoring
def evaluate_search(processor):
    query = "White Rabbit"
    search_results = processor.get_relevant_documents(query)

    # how many documents actually contins the query
    assessment = [ 1 if query in result.page_content else 0
                  for result in search_results]
    score = sum(assessment)
    print(score)
    return score

In [ ]:
import optuna
from optuna.study import StudyDirection

# function to be minimized
def objective(trial):
    splitter_service_id = trial.suggest_categorical('splitter_cls', configuration['splitter']['splitter_services'].keys())
    splitter_service = configuration['splitter']['splitter_services'][splitter_service_id]
    chunk_size = trial.suggest_int('chunk_size', 10, 8000)
    chunk_overlap = trial.suggest_int('chunk_overlap', 1, 1000)
    search_type = trial.suggest_categorical('search_type', configuration['search']['search_types'])
    k = trial.suggest_int('k', 1, 20)
    score_threshold = trial.suggest_float('score_threshold', 0.0, 1.0)
    return evaluate_search(get_processor_from_documennts(
        splitter_service, chunk_size, chunk_overlap, 
        search_type, k, score_threshold))

study = optuna.create_study(direction=StudyDirection.MAXIMIZE)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100)

study.best_params

# QA optimization

idem plus QA Evaluation

In [ ]:
#
#
#

In [ ]:
from math import ceil
from pathlib import Path

# import pytest
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader
from langchain.embeddings import FakeEmbeddings
from langchain.llms.fake import FakeListLLM
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS


# @pytest.mark.xfail
def test_all_in_one():
    # This is the source document.
    data_folder = "tests/unit/data"
    document_path = f"{data_folder}/hr_txt/carpool_policy.txt"

    file = Path(document_path)
    size = file.stat().st_size

    # Setup a text loader.
    loader = TextLoader(document_path)
    documents = loader.load()

    assert len(documents) == 1

    first_document = documents[0].page_content
    assert len(first_document) == size

    # Get your splitter ready.
    chunk_size = 1000
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=100)

    # Split your docs into texts.
    # assumes the chunk count is size / chunk size
    texts = text_splitter.split_documents(documents)
    nr_chunks = ceil(size / chunk_size)
    assert len(texts) == nr_chunks

    # Get embedding engine ready.
    # embeddings = OpenAIEmbeddings()
    embeddings = FakeEmbeddings(size=1352)

    # Embedd your texts andd store them in the vector database.
    # database is in memory.
    vector_store = FAISS.from_documents(texts, embeddings)
    assert len(vector_store.docstore._dict.keys()) == nr_chunks

    # Init a retriever for this db
    # lookup for some relevqnt parts
    nr_results = 1
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": nr_results})

    # retrieve some indexed documents relevant for this query
    query = "policy"
    docs = retriever.get_relevant_documents(query)
    assert len(docs) == nr_results

    # create a chain to answer questions with this documents
    responses = ["Action: Policy understanding", "Final Answer: answer"]

    qa = RetrievalQA.from_chain_type(
        llm=FakeListLLM(responses=responses), chain_type="stuff", retriever=retriever, return_source_documents=True
    )

    response = qa({"query": query})

    assert response["result"] == responses[0]


# @pytest.mark.xfail
def test_update_vector_store():
    # This are the source documents.
    documents_folder = "tests/unit/data/hr_txt"
    documents_names = ["carpool_policy.txt", "holidays_policy.txt"]

    # Get your splitter ready.
    chunk_size = 1000
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=100)

    # Get embedding engine ready.
    # embeddings = OpenAIEmbeddings()
    embeddings = FakeEmbeddings(size=1352)

    vector_store = None

    for document_name in documents_names:
        document_path = f"{documents_folder}/{document_name}"
        file = Path(document_path)
        size = file.stat().st_size

        # Setup a text loader.
        loader = TextLoader(document_path)
        documents = loader.load()
        assert len(documents) == 1

        first_document = documents[0].page_content
        assert len(first_document) == size

        # Split your docs into texts.
        # assumes the chunk count is size / chunk size
        texts = text_splitter.split_documents(documents)
        nr_chunks = ceil(size / chunk_size)
        expected_nr_chunks = (nr_chunks, nr_chunks + 1)
        assert len(texts) in expected_nr_chunks

        # Embedd your texts andd store them in the vector database.
        # database is in memory.
        if vector_store:
            nr_before = len(vector_store.docstore._dict.keys())
            # vector_store.add_texts(texts)
            vector_store.add_documents(texts)
            nr_after = nr_chunks + nr_before
            expected_nr_chunks = (nr_after, nr_after + 1, nr_after + 2)
            assert len(vector_store.docstore._dict.keys()) in expected_nr_chunks

        else:
            vector_store = FAISS.from_documents(texts, embeddings)
            assert len(vector_store.docstore._dict.keys()) in expected_nr_chunks


# @pytest.mark.xfail
def test_metadata():
    # This is the source document.
    data_folder = "tests/unit/data"
    document_path = f"{data_folder}/hr_txt/carpool_policy.txt"

    file = Path(document_path)
    size = file.stat().st_size

    # Setup a text loader.
    loader = TextLoader(document_path)
    documents = loader.load()

    assert len(documents) == 1

    first_document = documents[0].page_content
    assert len(first_document) == size

    # set metadata
    for document in documents:
        document.metadata = {"path": document_path}

    # Get your splitter ready.
    chunk_size = 1000
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=100)

    # Split your docs into texts.
    # assumes the chunk count is size / chunk size
    texts = text_splitter.split_documents(documents)
    nr_chunks = ceil(size / chunk_size)
    assert len(texts) == nr_chunks

    first_test = texts[0]
    assert "path" in first_test.metadata
    assert isinstance(first_test, Document)
    assert first_test.metadata["path"] == document_path

    # Get embedding engine ready.
    # embeddings = OpenAIEmbeddings()
    embeddings = FakeEmbeddings(size=1352)

    # Embedd your texts andd store them in the vector database.
    # database is in memory.
    # split actually returns Langchain's Document objects
    # it keeps track of metadata
    vector_store = FAISS.from_documents(texts, embeddings)
    assert len(vector_store.docstore._dict.keys()) == nr_chunks

    # Init a retriever for this db
    # lookup for some relevqnt parts
    nr_results = 1
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": nr_results})

    # retrieve some indexed documents relevant for this query
    query = "policy"
    docs = retriever.get_relevant_documents(query)
    assert len(docs) == nr_results

    # create a chain to answer questions with this documents
    responses = ["Action: Policy understanding", "Final Answer: answer"]

    qa = RetrievalQA.from_chain_type(
        llm=FakeListLLM(responses=responses), chain_type="stuff", retriever=retriever, return_source_documents=True
    )

    response = qa({"query": query})

    assert response["result"] == responses[0]
    assert "source_documents" in response.keys()
    assert len(response["source_documents"]) == 1

    first_source = response["source_documents"][0]
    assert isinstance(first_source, Document)
    assert first_source.metadata["path"] == document_path


In [ ]:
    def get_sources_with_similarity_and_score(self, user_question: str) -> Dict[Any, float]:
        results_with_scores = self.vector_store.similarity_search_with_score(
            user_question,
            k=self.llm_context["retriever_k"],
            score_threshold=self.llm_context["retriever_score_threshold"],
        )
        # for doc, score in results_with_scores:
        #    logger.info(f"Content: {doc.page_content}, Metadata: {doc.metadata}, Score: {score}")
        # warning - eats the iterator
        return results_with_scores

    def get_sources_with_mmr_and_score(self, user_question: str) -> List[Any]:
        results = self.vector_store.max_marginal_relevance_search(user_question)
        # for doc in results:
        #    logger.into(f"Content: {doc.page_content}, Metadata: {doc.metadata}")
        # warning - eats the iterator
        return results

    def get_sources_with_retriever(self, user_question: str) -> List[Any]:
        results = self.retriever.get_relevant_documents(user_question)
        # for doc in results:
        #    logger.info(f"Content: {doc.page_content}, Metadata: {doc.metadata}")
        # warning - eats the iterator
        return results
